### Day 12 - 模块与高级特性**前置**:Day01-11 全部语法点(print/input/变量/字符串/分支/循环/函数/列表字典/文件 I/O/异常/OOP 封装继承多态组合)**关键问题**:代码破千行后,单文件不堪重负 —— 怎么拆分到多个文件、让重复的计时/过滤/转换逻辑自动复用。**本日目标**:掌握模块 import / 包结构 / 生成器 yield / lambda 匿名函数 /map filter reduce / 装饰器 @wraps / 迭代器协议

**引入**Day07-10 我们用 OOP 把数据与行为打包成对象,但对象散落在**一个文件**里,上千行代码翻来翻去很痛苦。Day12 解决三个新问题:1. 怎么拆文件 —— 模块 + 包,让代码按功能分文件夹存放2. 怎么写更短 —— lambda / map / filter 让一行逻辑不需要 def3. 怎么自动复用 —— 装饰器让计时/日志等横切逻辑一处定义处处生效

#### 知识点 1:模块导入 import / from / as**痛点**:一个 .py 写到 1000 行,翻来翻去找函数非常痛苦,命名还容易冲突。**类比**:模块就像抽屉 —— 把不同工具分门别类放进各自的抽屉,用到哪个拉开哪个,不需要全堆在桌面上。**解释**:Python 把每个 .py 文件看作一个**模块**,用 import 即可把另一个文件的函数/变量/类引入当前文件。三种写法:- import 模块名 -> 引入整个模块,通过 模块名.成员 访问- from 模块名 import 成员 -> 直接把成员拿过来用,无需前缀- import 模块名 as 别名 -> 给长名字起短名,防止命名冲突

> **ASCII 内存图:命名空间**> ```> 当前文件命名空间            math 模块内部> +--------------+          +----------------+> | sqrt --------+--------->| sqrt 函数      |> | pi  ---------+--------->| pi = 3.14159.. |> +--------------+          +----------------+>> from math import sqrt 之后:> 当前命名空间里多了一个 sqrt,> 调用时直接写 sqrt(16),不再需要 math 前缀> ```

In [ ]:
# 导入 Python 标准库 math 模块import math# 通过 模块名.成员名 访问函数和常量result = math.sqrt(16)pi = math.piprint(result)               # 4.0print(pi)                   # 3.141592653589793# --- 执行过程 ---# 第 1 行 import math:#   ① Python 在 sys.path 中查找 math 模块#   ② 首次导入时执行模块代码,创建模块对象#   ③ 把模块对象绑定到变量 math## 第 4 行 math.sqrt(16):#   ① 从 math 模块中找到 sqrt 函数#   ② 调用 sqrt(16) -> 返回 4.0## 第 5 行 math.pi:#   ① 从 math 模块中读取 pi -> 3.14159...## 注意:重复 import 不会重新执行模块,模块只加载一次

In [ ]:
# from ... import:直接把成员引入当前命名空间from math import sqrt, pi# 调用时直接写名字,不需要 math. 前缀print(sqrt(25))             # 5.0print(pi)                   # 3.141592653589793# --- 执行过程 ---# 第 1 行 from math import sqrt, pi:#   ① 找到 math 模块#   ② 从模块中取出 sqrt 和 pi#   ③ 把这两个名字绑定到当前命名空间## 第 4 行 sqrt(25):#   ① 在当前命名空间找到 sqrt(不是 math.sqrt)#   ② 调用 -> 返回 5.0

In [ ]:
# import as:给模块或成员起别名import math as mfrom math import factorial as facprint(m.sqrt(9))            # 3.0print(fac(5))               # 120# --- 执行过程 ---# 第 2 行 import math as m:#   ① 导入 math 模块#   ② 在当前命名空间绑定变量 m,指向 math 模块#   ③ 之后 m.sqrt 等同于 math.sqrt## 第 3 行 from math import factorial as fac:#   ① 从 math 模块取出 factorial 函数#   ② 绑定到变量 fac

> **逐行解剖**

> - import math:全量导入,通过 math.xxx 访问,不会和当前名字冲突> - from math import sqrt:按需导入,直接用 sqrt(),当前有同名会被覆盖> - import math as m:给模块起别名,常用于长名模块(如 numpy as np)

> **常见错误**> 1. **错误现象**:ImportError: No module named mymodule>    **原因**:mymodule.py 不在当前目录或 sys.path 中的任何路径> 2. **错误现象**:NameError: name x is not defined>    **原因**:用了 from x import y,但 x 模块里根本没有 y 这个成员

> 问自己:> - import math 和 from math import sqrt 在命名空间上有什么差别?> - 如果当前文件也有一个 sqrt 函数,from math import sqrt 后哪个生效?> - 重复写两次 import math,Python 会执行两遍模块代码吗?

In [ ]:
# ======================# 学员代码区# ======================# 从 random 模块导入 randint,生成 [1, 100] 的随机整数并打印from random import randintn = randint(1, 100)# print(n)pass

In [ ]:
# ======================# 参考答案# ======================from random import randintn = randint(1, 100)print(n)

> **小结**:三种导入方式各有用途。团队协作常见约定:> numpy as np、pandas as pd 都是 as 别名,> 长模块名用别名让代码更短。

#### 知识点 2:包结构 __init__.py**痛点**:模块多了散落在目录里,没有一个总入口,导入路径很长。**类比**:包就像工具箱 —— 零散模块(螺丝刀、锤子)装进一个有标签的箱子,打开箱子就能按标签找到工具。**解释**:包 = 带 __init__.py 的文件夹,里面放多个模块。__init__.py在包被导入时自动执行,可以用来做初始化、控制导出列表。__all__列表决定 from 包 import * 时暴露哪些模块。

> **ASCII 目录结构**> ```> myutils/                     <- 包目录> |-- __init__.py              <- 包初始化文件(必须存在)> |-- math_tools.py            <- 子模块:数学工具> +-- string_tools.py          <- 子模块:字符串工具>> __init__.py 内容:>   __all__ = ["math_tools", "string_tools"]>> 使用:>   from myutils.math_tools import circle_area> ```

In [ ]:
# 模拟包导入的执行流程(用注释说明目录结构)# 目录结构:# mypkg/#   __init__.py      -> 写 __all__ = ["math_tools"]#   math_tools.py    -> 写 def add(a,b): return a+b## 实际使用时:#   from mypkg.math_tools import add#   print(add(2, 3))# __init__.py 的作用演示print("包导入时,会先执行 __init__.py")print("包导入后,可按 包.子模块.成员 的方式逐层访问")# --- 执行过程 ---# from mypkg import math_tools:#   ① Python 先找到 mypkg 目录#   ② 执行 mypkg/__init__.py(如有初始化逻辑会自动跑)#   ③ 找到 mypkg/math_tools.py 并加载#   ④ 把 math_tools 模块绑定到当前命名空间

> **逐行解剖**

> - __init__.py:可以是空文件,但必须存在,Python 才认为目录是包> - __all__ = ["math_tools"]:控制 from mypkg import * 暴露的模块列表> - from mypkg.math_tools import add:逐层访问,包 -> 子模块 -> 成员

> **常见错误**> 1. **错误现象**:ModuleNotFoundError: No module named mypkg>    **原因**:引用包的 .py 文件和 mypkg 文件夹不在同一目录> 2. **错误现象**:ImportError: cannot import name xxx from mypkg>    **原因**:__all__ 里没有列 xxx,或该成员根本不存在

> 问自己:> - 如果 __init__.py 是空文件,Python 是否还把这个目录当作包?> - __all__ 不写的话,from 包 import * 会导入哪些内容?> - 包的嵌套(包里还有包)应该怎么写导入路径?

In [ ]:
# ======================# 学员代码区# ======================# 假设有一个包 mycalc,结构如下:# mycalc/#   __init__.py     -> __all__ = ["basic"]#   basic.py        -> def mul(a,b): return a*b# 请从 mycalc 包导入 mul 函数,计算 7*8pass

In [ ]:
# ======================# 参考答案# ======================# 先创建目录结构:#   mycalc/__init__.py: __all__ = ["basic"]#   mycalc/basic.py:    def mul(a,b): return a*b# 然后:from mycalc.basic import mulprint(mul(7, 8))            # 56

#### 知识点 3:生成器函数 yield**痛点**:有 100 万个数据要处理,一次性读到列表里会撑爆内存。**类比**:生成器就像按需点餐 —— 吃一盘上一盘,不像列表那样把菜全端上来。**解释**:包含 yield 的函数就是**生成器函数**,调用它不会立即执行函数体,而是返回一个生成器对象。每次 next() 或在 for 循环中,函数执行到 yield就暂停并返回一个值,下次再从暂停处继续。

In [ ]:
def squares(n):    # 生成器函数:用 yield 逐个产出平方数    for i in range(n):        yield i * i       # 暂停并产出 i 的平方# 调用 squares(4):不执行函数体,只返回生成器对象gen = squares(4)print(gen)                  # <generator object squares at 0x...># for 循环内部反复调用 next(),逐个取值gen2 = squares(4)for v in gen2:    print(v)                # 0  1  4  9# --- 执行过程 ---# 第 6 行 gen = squares(4):#   ① 调用 squares 函数,但不执行函数体#   ② 返回生成器对象赋给 gen## for v in gen2 第 1 次迭代:#   ① 调用 next(gen2) -> 进入函数,i=0#   ② 遇到 yield 0 -> 暂停,返回 0#   ③ v 被赋值为 0## for v in gen2 第 2 次迭代:#   ① 调用 next(gen2) -> 从暂停处继续,i=1#   ② yield 1 -> 暂停,返回 1## for 遇到 StopIteration -> 结束循环

In [ ]:
def fib(n):    # 产出前 n 个斐波那契数    a, b = 0, 1              # 前两个初始值    for _ in range(n):        yield a              # 产出当前值        a, b = b, a + b      # 同步更新 a 和 b# 手动用 next() 推进gen = fib(5)print(next(gen))            # 0print(next(gen))            # 1print(next(gen))            # 1# --- 执行过程 ---# gen = fib(5):#   ① 返回生成器对象,不执行函数体## 第 1 次 next(gen):#   ① 进入函数,a=0,b=1#   ② yield a -> 产出 0#   ③ 暂停在此处,返回 0## 第 2 次 next(gen):#   ① 从暂停处继续:执行 a, b = b, a+b -> a=1,b=1#   ② 回到 for 头部,循环继续#   ③ yield a -> 产出 1

> **逐行解剖**

> - yield i * i:把 i*i 产出给调用者,函数在此处暂停> - 再次被唤醒时,从 yield 下一行(即 for 的下一个 i)继续> - for 循环耗尽后,函数自然结束,自动抛出 StopIteration

> **常见错误**> 1. **错误现象**:TypeError: object of type generator has no len()>    **原因**:生成器不能 len(),因为它不一次性生成所有值> 2. **错误现象**:第二次 for 遍历生成器结果为空>    **原因**:生成器只能遍历一次,遍历完就空了

> 问自己:> - 生成器和普通函数在调用时最本质的区别是什么?> - 生成器能 gen[0] 这样按索引取第 5 个值吗?> - 什么场景下一定不能用列表而必须用生成器?

In [ ]:
# ======================# 学员代码区# ======================# 写一个生成器 countdown(n),从 n 倒数到 1# count = countdown(5) -> 5, 4, 3, 2, 1def countdown(n):    pass

In [ ]:
# ======================# 参考答案# ======================def countdown(n):    while n > 0:        yield n        n -= 1# 验证print(list(countdown(5)))   # [5, 4, 3, 2, 1]

#### 知识点 4:生成器表达式**痛点**:列表推导 [x*x for x in range(1000000)] 一次占几十 MB。**类比**:列表推导是一次炒 1000 份菜,生成器表达式是点一份炒一份。**解释**:语法和列表推导几乎一样,只是把 [] 换成 ()。返回的是生成器对象,逐个产出值,不一次性占内存。常用于传给 sum() / max() 等函数作为参数,这时括号可以省略: sum(x*x for x in range(n))。

In [ ]:
import sys# 列表推导:一次算出所有值存入内存lst = [x * x for x in range(1000)]print(type(lst))            # <class list>print(sys.getsizeof(lst))   # 约 8856 字节# 生成器表达式:逐个产出,几乎不占内存gen = (x * x for x in range(1000))print(type(gen))            # <class generator>print(sys.getsizeof(gen))   # 约 200 字节# --- 执行过程 ---# 列表推导:对 range(1000) 每个元素求平方#   -> 内存中创建一个含 1000 个元素的 list## 第 8 行 gen = (x*x for x in range(1000)):#   ① 不立即计算任何元素#   ② 只生成一个生成器对象,内部记录表达式和范围## 当代码遍历 gen 时,才逐个计算并产出

In [ ]:
# 生成器表达式作为函数参数时,括号可省略# 计算 0~9 的平方和total = sum(x * x for x in range(10))print(total)                # 285# 筛选偶数后求和evens = sum(x for x in range(10) if x % 2 == 0)print(evens)                # 20# --- 执行过程 ---# sum(x*x for x in range(10)):#   ① sum 内部对生成器迭代#   ② 第 1 次取到 0,累加#   ③ 第 2 次取到 1,累加#   ④ ...直到生成器耗尽#   ⑤ 返回累加结果 285

> **逐行解剖**

> - (x*x for x in range(10)):生成器表达式,用圆括号> - sum(x*x for x in range(10)):作为唯一参数时外层括号可省> - if x % 2 == 0:生成器表达式也支持过滤子句

> **常见错误**> 1. **错误现象**:操作后发现 gen 是空的>    **原因**:之前遍历过一遍,生成器已经耗尽> 2. **错误现象**:SyntaxError>    **原因**:把圆括号写成方括号,变成了列表推导

> 问自己:> - 列表推导和生成器表达式运行时的主要区别是什么?> - 生成器表达式能多次使用吗?> - 什么情况下用列表推导,什么情况下用生成器表达式?

In [ ]:
# ======================# 学员代码区# ======================# 用生成器表达式计算 1~20 所有奇数的和pass

In [ ]:
# ======================# 参考答案# ======================total = sum(x for x in range(1, 21) if x % 2 == 1)print(total)                # 100

#### 知识点 5:lambda 匿名函数**痛点**:一个简单的两数相加也要写 3 行 def,代码噪音太大。**类比**:lambda 就像一次性纸杯 —— 用完即扔,适合简单场景,不值得专门给它取名字(好比不值得买个保温杯喝一口水)。**解释**:lambda 参数: 表达式 定义一个匿名函数,自动返回表达式结果。语法限制:**只能写一个表达式**,不能有赋值、循环、return 多个语句。适合传给 sorted() / map() 等需要短函数的地方。

In [ ]:
# 用 def 定义普通函数def add(x, y):    return x + yprint(add(2, 3))            # 5# 用 lambda 定义等价的匿名函数add_lambda = lambda x, y: x + yprint(add_lambda(2, 3))     # 5# lambda 常用作 sorted 的 key 参数words = ["banana", "apple", "cherry"]# 按字符串长度排序words.sort(key=lambda s: len(s))print(words)                # apple banana cherry# --- 执行过程 ---# 第 6 行 add_lambda = lambda x, y: x + y:#   ① 创建一个函数对象,接收两个参数 x, y#   ② 自动返回 x + y(不需要写 return)#   ③ 把函数对象赋给变量 add_lambda## 第 12 行 sort(key=lambda s: len(s)):#   ① sort 对每个元素调用 lambda s: len(s)#   ② 得到每个字符串的长度#   ③ 按长度升序排列

In [ ]:
# lambda 的典型场景:作为回调传给高阶函数nums = [1, 2, 3, 4, 5]# 用 lambda 做简单的映射函数squares = list(map(lambda x: x * x, nums))print(squares)              # [1, 4, 9, 16, 25]# lambda 与 sorted 结合students = [    {"name": "张三", "score": 88},    {"name": "李四", "score": 95},    {"name": "王五", "score": 76},]# 按成绩降序top = sorted(students, key=lambda s: s["score"], reverse=True)for s in top:    print(s["name"], s["score"])# --- 执行过程 ---# sorted 的 key 参数:#   ① 对每个元素调用 lambda,取出 score#   ② 按 score 数值排序,reverse=True 降序

> **逐行解剖**

> - lambda x, y: x + y:冒号前是参数,冒号后是返回的表达式> - 只能有一个表达式,不能写多行语句> - 通常作为参数直接传给高阶函数,用完即弃

> **常见错误**> 1. **错误现象**:SyntaxError>    **原因**:lambda 里写了 x = 1 等赋值语句,lambda 只能有表达式> 2. **错误现象**:过几个月回头看不懂>    **原因**:逻辑复杂时滥用 lambda 导致可读性差,应改用 def

> 问自己:> - lambda 能替代所有 def 函数吗?什么情况下不能?> - lambda x: x % 2 == 0 返回的是什么类型的值?> - 为什么不把所有函数都写成 lambda?

In [ ]:
# ======================# 学员代码区# ======================# 写一个 lambda 函数 is_even,判断一个数是否为偶数is_even = Nonepass

In [ ]:
# ======================# 参考答案# ======================is_even = lambda x: x % 2 == 0print(is_even(4))           # Trueprint(is_even(7))           # False

#### 知识点 6:map / filter / reduce / sorted**痛点**:对列表逐个转换/筛选要写 for 循环,代码啰嗦。**类比**:map = 流水线给每个零件打磨,filter = 质检员筛掉次品,reduce = 把所有零件组装成一件成品。**解释**:- map(fn, iterable):对每个元素调用 fn,返回结果的迭代器- filter(fn, iterable):只保留 fn 返回 True 的元素- reduce(fn, iterable):从左到右累积- sorted(iterable, key=fn):返回排序后的新列表

In [ ]:
from functools import reducenums = [1, 2, 3, 4, 5]# map:每个元素乘 2doubled = list(map(lambda x: x * 2, nums))print(doubled)              # [2, 4, 6, 8, 10]# filter:只保留偶数evens = list(filter(lambda x: x % 2 == 0, nums))print(evens)                # [2, 4]# reduce:求乘积product = reduce(lambda acc, x: acc * x, nums)print(product)              # 120# --- 执行过程 ---# map 处理流程:#   ① 第 1 次取 1 -> 1*2 -> 2#   ② 第 2 次取 2 -> 2*2 -> 4#   ③ ...直到列表耗尽## reduce 处理流程:#   ① acc=1, x=2 -> 1*2=2#   ② acc=2, x=3 -> 2*3=6#   ③ acc=6, x=4 -> 6*4=24#   ④ acc=24, x=5 -> 24*5=120

In [ ]:
# sorted:按指定 key 排序students = [    {"name": "张三", "score": 88},    {"name": "李四", "score": 95},    {"name": "王五", "score": 76},]# 按成绩升序by_score = sorted(students, key=lambda s: s["score"])print([s["name"] for s in by_score])# --- 执行过程 ---# sorted 的 key 函数:#   ① 对每个元素调用 lambda,提取排序依据#   ② Python 依据这些 key 值做比较#   ③ 返回新列表,原列表不变

> **逐行解剖**

> - map(fn, iterable):返回 map 对象(迭代器),需 list() 转成列表> - filter(fn, iterable):fn 返回 True 才保留> - reduce(fn, iterable):需要 from functools import reduce> - sorted(key=fn):key 函数只返回比较依据,不修改原元素

> **常见错误**> 1. **错误现象**:NameError: name reduce is not defined>    **原因**:忘记 from functools import reduce> 2. **错误现象**:以为 map/filter 返回 list,做下标访问报错>    **原因**:map/filter 返回的是迭代器,需要 list() 才能索引

> 问自己:> - map 和列表推导 [f(x) for x in lst] 哪个更清晰?> - reduce 的初始值怎么指定?> - sorted 和 list.sort() 有什么区别?

In [ ]:
# ======================# 学员代码区# ======================# names = ["Alice", "Bob", "Charlie"]# 1. 用 map 把所有名字转大写# 2. 用 filter 筛选出长度 > 3 的名字from functools import reducenames = ["Alice", "Bob", "Charlie"]pass

In [ ]:
# ======================# 参考答案# ======================uppered = list(map(lambda s: s.upper(), names))print(uppered)              # ALICE BOB CHARLIElong = list(filter(lambda s: len(s) > 3, names))print(long)                 # Alice Charlie

#### 知识点 7:装饰器 @decorator + @wraps + 计时器**痛点**:想给多个函数加计时逻辑,每个函数手动改就是复制粘贴。**类比**:装饰器就像包装礼物 —— 不改变礼物本身,在外面多包一层包装纸。**解释**:装饰器本质是**接收函数、返回新函数**的高阶函数。@decorator 语法糖等价于 func = decorator(func)。@functools.wraps(fn) 保留原函数的 __name__ 等元信息,对调试很重要。

In [ ]:
import timeimport functoolsdef timer(fn):    # 装饰器:给函数加计时,不修改原函数    @functools.wraps(fn)     # 保留原函数的 __name__ 等元信息    def wrapper(*args, **kwargs):        start = time.time()        result = fn(*args, **kwargs)   # 调用原函数        elapsed = time.time() - start        print(f"{fn.__name__} 耗时 {elapsed:.4f} 秒")        return result    return wrapper@timerdef slow_sum(n):    total = 0    for i in range(n):        total += i    return totalprint(slow_sum(1000000))# --- 执行过程 ---# @timer 等价于: slow_sum = timer(slow_sum)#   ① 调用 timer(slow_sum)#   ② 返回 wrapper 函数#   ③ 变量 slow_sum 现在指向 wrapper## slow_sum(1000000):#   ① 实际调用 wrapper(1000000)#   ② 记录 start 时间#   ③ 调用原 slow_sum(1000000) -> 得到结果#   ④ 计算耗时并打印#   ⑤ 返回结果

In [ ]:
# 演示 @wraps 的重要性import functoolsdef without_wraps(fn):    # 没加 @wraps 的装饰器    def wrapper(*args, **kwargs):        return fn(*args, **kwargs)    return wrapperdef with_wraps(fn):    # 加了 @wraps 的装饰器    @functools.wraps(fn)    def wrapper(*args, **kwargs):        return fn(*args, **kwargs)    return wrapper@without_wrapsdef greet1():    pass@with_wrapsdef greet2():    passprint(greet1.__name__)       # wrapper (元信息丢失!print(greet2.__name__)       # greet2 (保留)# --- 执行过程 ---# 没有 @wraps 时:#   greet1.__name__ -> wrapper (被替换了)# 有 @wraps 时:#   greet2.__name__ -> greet2 (原样保留)

> **逐行解剖**

> - def timer(fn):装饰器函数,接收被装饰的函数> - @functools.wraps(fn):把原函数的元信息复制到 wrapper> - def wrapper(*args, **kwargs):适配任意参数签名> - @timer:语法糖,等价于 slow_sum = timer(slow_sum)

> **常见错误**> 1. **错误现象**:TypeError: wrapper() takes 0 positional arguments>    **原因**:wrapper 参数签名写死,没写 *args, **kwargs> 2. **错误现象**:调试时函数名全变成 wrapper>    **原因**:忘记加 @functools.wraps(fn),元信息丢失

> 问自己:> - 如果 wrapper 不写 return fn(...),被装饰函数还能拿到返回值吗?> - 多个装饰器叠加时,执行顺序是怎样的?> - 装饰器能装饰类的方法吗?需要注意什么?

In [ ]:
# ======================# 学员代码区# ======================# 写一个 @log 装饰器,每次调用时打印函数名和参数import functoolsdef log(fn):    pass# 测试函数# @log# def add(a, b):#     return a + b

In [ ]:
# ======================# 参考答案# ======================def log(fn):    @functools.wraps(fn)    def wrapper(*args, **kwargs):        print(f"调用 {fn.__name__}, args={args}, kwargs={kwargs}")        return fn(*args, **kwargs)    return wrapper@logdef add(a, b):    return a + bprint(add(2, 3))            # 先打印日志,再输出 5

#### 知识点 8:迭代器协议 __iter__ / __next__**痛点**:想让自定义的类支持 for 循环,不知道要写什么方法。**类比**:迭代器就像翻页器 —— 每次翻一页(__next__),没有下一页就停下来(StopIteration),而可翻页这个能力本身由 __iter__ 声明。**解释**:Python 的 for 循环依赖**迭代器协议**:- __iter__(self):返回迭代器对象(通常返回 self)- __next__(self):返回下一个值,没有就 raise StopIteration任何实现了这两个方法的对象都可以被 for 循环遍历。

In [ ]:
class CountDown:    # 自定义迭代器:从 n 倒数到 1    def __init__(self, n):        self.n = n           # 当前值    def __iter__(self):        # 返回迭代器对象(自己就是迭代器)        return self    def __next__(self):        # 返回下一个值,没有就抛 StopIteration        if self.n <= 0:            raise StopIteration        current = self.n        self.n -= 1        return current# 使用自定义迭代器cd = CountDown(5)for v in cd:    print(v)                # 5 4 3 2 1# --- 执行过程 ---# for v in CountDown(5):#   ① 调用 cd.__iter__() -> 返回 self#   ② 调用 __next__() -> n=5>0,返回 5,n 变为 4#   ③ 调用 __next__() -> n=4>0,返回 4,n 变为 3#   ④ ...直到 n=0,raise StopIteration#   ⑤ for 捕获 StopIteration,结束循环

In [ ]:
# 用生成器函数实现同样的逻辑(更简洁)def countdown_gen(n):    while n > 0:        yield n        n -= 1# 生成器自动实现 __iter__ 和 __next__for v in countdown_gen(5):    print(v)                # 5 4 3 2 1# --- 执行过程 ---# countdown_gen(5) 返回生成器对象# 生成器对象自动拥有 __iter__ 和 __next__# for 循环内部调用 __next__ 直到 StopIteration

> **逐行解剖**

> - __iter__:for 循环开始时调用,返回迭代器对象> - __next__:每次迭代调用,返回下一个值> - raise StopIteration:告诉 for 循环没有更多元素了> - 生成器函数(yield)自动实现这两个方法,所以更常用

> **常见错误**> 1. **错误现象**:TypeError: X object is not iterable>    **原因**:for 循环的对象没有实现 __iter__ 方法> 2. **错误现象**:死循环>    **原因**:__next__ 永远不 raise StopIteration

> 问自己:> - 可迭代对象(iterable)和迭代器(iterator)有什么区别?> - 为什么 __iter__ 要返回 self 而不是别的?> - 生成器函数和手写迭代器类,哪个更常用?为什么?

In [ ]:
# ======================# 学员代码区# ======================# 写一个 RangeIterator 类,模拟 range(stop) 的功能# 迭代时产出 0, 1, 2, ..., stop-1class RangeIterator:    pass

In [ ]:
# ======================# 参考答案# ======================class RangeIterator:    def __init__(self, stop):        self.stop = stop        self.current = 0    def __iter__(self):        return self    def __next__(self):        if self.current >= self.stop:            raise StopIteration        val = self.current        self.current += 1        return val# 验证for i in RangeIterator(5):    print(i, end=" ")       # 0 1 2 3 4

**今日小结**| 概念 | 解决的痛点 | 核心语法 ||---|---|---|| 模块导入 | 单文件太长,命名冲突 | import / from / as || 包结构 | 模块多了散乱 | __init__.py / __all__ || 生成器函数 | 大数据撑爆内存 | yield || 生成器表达式 | 列表推导占内存 | (x for x in ...) || lambda | 简单逻辑也要 def | lambda x: x + 1 || map/filter/reduce | 逐个转换/筛选啰嗦 | map(f, lst) || 装饰器 | 横切逻辑重复 | @decorator + @wraps || 迭代器协议 | 自定义类支持 for | __iter__ / __next__ |共同主题:让代码更短、更省内存、更易复用。

**更多练习**- 当堂练:course/lesson12/in_class/practice01.py ~ practice08.py- 课后作业:course/lesson12/homework/task01.py ~ task03.py